In [1]:
import pandas as pd
import altair as alt

if not hasattr(alt.utils, 'use_signature_func'):
    def _use_signature_func(_signature_func):
        def _decorator(func):
            return func
        return _decorator

    alt.utils.use_signature_func = _use_signature_func
import os

from ecostyles import EcoStyles
# Create styles instance
styles = EcoStyles()
# Register and enable a theme
styles.register_and_enable_theme(theme_name="article")

In [2]:
hours_wide = pd.DataFrame({
    'Period': list(range(1997, 2024)),
    'Current approach': [31.7, 32.2, 32.1, 31.9, 31.9, 31.6, 31.4, 31.1, 31.5, 31.4,
                         31.5, 31.1, 31.1, 30.9, 31.2, 31.4, 31.7, 31.8, 31.5, 31.8,
                         31.8, 31.8, 31.8, 28.8, 31.1, 31.8, 31.6],
    'Component approach': [29.4, 29.8, 29.5, 29.4, 29.4, 29.2, 29.1, 29.2, 28.9, 29.1,
                           29.0, 28.9, 28.6, 28.5, 28.2, 28.2, 28.3, 28.3, 28.4, 28.4,
                           28.2, 28.1, 28.0, 24.0, 26.5, 27.3, 27.3],
})

# Rename series to the labels used on the chart

series_names = {'Current approach': 'Old Method', 'Component approach': 'New Method'}

# Wide to long
hours_long = (
    hours_wide.melt(id_vars='Period', var_name='Method', value_name='Hours')
              .assign(Method=lambda d: d['Method'].map(series_names))
)

hours_long.head()

,Period,Method,Hours
0,1997,Old Method,31.7
1,1998,Old Method,32.2
2,1999,Old Method,32.1
3,2000,Old Method,31.9
4,2001,Old Method,31.9


In [3]:
# --- Scales and colours ---
order  = ['New Method', 'Old Method']
colors = ['#179fdb', '#122b39']          # eco blue for the new series, eco navy for the old
color_scale = alt.Scale(domain=order, range=colors)

x_scale = alt.Scale(domain=[1997, 2027], nice=False)   # extra room on the right for end labels
y_scale = alt.Scale(domain=[22, 34], zero=False)

# --- Main lines ---
lines = alt.Chart(hours_long).mark_line(strokeWidth=2.5).encode(
    x=alt.X('Period:Q', scale=x_scale,
            axis=alt.Axis(format='d', grid=False, title=None, labelFontSize=14, labelAngle=0,
                          values=[2000, 2005, 2010, 2015, 2020])),
    y=alt.Y('Hours:Q', scale=y_scale,
            axis=alt.Axis(title='Average weekly hours worked', titleFontSize=14,
                          labelFontSize=14, values=list(range(22, 35, 2)),
                          titleAngle=0, titleAlign='left', titleBaseline='bottom',
                          titleX=-30, titleY=-10)),
    color=alt.Color('Method:N', scale=color_scale, legend=None),
    tooltip=[
        alt.Tooltip('Method:N', title='Method'),
        alt.Tooltip('Period:Q', title='Year', format='d'),
        alt.Tooltip('Hours:Q', title='Average weekly hours', format='.1f'),
    ]
)

# --- End-of-line labels (adjust dy to nudge labels up/down; negative = up) ---
end_dy = {
    'New Method': 0,
    'Old Method': 0,
}
end_pts = hours_long.loc[hours_long.groupby('Method')['Period'].idxmax()]

end_labels = alt.layer(*[
    alt.Chart(end_pts[end_pts['Method'] == m]).mark_text(
        align='left', dx=6, dy=end_dy[m], fontSize=14, fontWeight='bold'
    ).encode(
        x=alt.X('Period:Q', scale=x_scale),
        y=alt.Y('Hours:Q', scale=y_scale),
        text='Method:N',
        color=alt.Color('Method:N', scale=color_scale, legend=None)
    )
    for m in order
])

# --- 2008 threshold rule (behind the data) ---
crisis = pd.DataFrame({'Period': [2008], 'label': ['2008 Crisis']})

crisis_rule = alt.Chart(crisis).mark_rule(
    color='#676A86', strokeDash=[5, 4], strokeWidth=1.5, opacity=0.8
).encode(x=alt.X('Period:Q', scale=x_scale))

crisis_label = alt.Chart(crisis).mark_text(
    align='left', baseline='top', dx=5, dy=0, fontSize=13, color='#676A86'
).encode(
    x=alt.X('Period:Q', scale=x_scale),
    y=alt.value(5),
    text='label:N'
)

# --- Pandemic annotation at the New Method low point ---
# Adjust these to move the annotation around
pandemic_dx, pandemic_dy = -8, 0

low_point = hours_long[hours_long['Method'] == 'New Method'].nsmallest(1, 'Hours').assign(
    label='Pandemic and furlough'
)

pandemic_label = alt.Chart(low_point).mark_text(
    align='right', baseline='middle', dx=pandemic_dx, dy=pandemic_dy,
    fontSize=13, fontStyle='italic', color='#179fdb'
).encode(
    x=alt.X('Period:Q', scale=x_scale),
    y=alt.Y('Hours:Q', scale=y_scale),
    text='label:N'
)

# --- Combine ---
hours_chart = (
    alt.layer(crisis_rule, crisis_label, lines, end_labels, pandemic_label)
       .properties(width=560, height=340)
       .configure_view(strokeWidth=0)
)

hours_chart

alt.LayerChart(...)

In [4]:
# Save to png
hours_chart.save('average_weekly_hours.png', scale_factor=2)
# Save to json
hours_chart.save('average_weekly_hours.json', scale_factor=2)